# Pseudo-Log-Likelihood (PLL) Bias Analysis — LLaMA-2 (English)

This notebook computes PLL-based bias scores for **LLaMA-2** on the StereoSet dataset (720 instances).

**Metrics computed:**
- PLL per sentence (stereotype, anti-stereotype, unrelated)
- BiasScore = PLL(stereotype) − PLL(anti-stereotype)
- Categorical Bias (CB) per bias type

**Bias categories:** gender, race, religion, profession

In [ ]:
!pip install peft accelerate bitsandbytes transformers -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import torch
import math
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
!huggingface-cli login

In [ ]:
# Load dataset
df = pd.read_excel('/content/drive/MyDrive/NewStreoSet_Dataset.xlsx')
print(df.shape)
df.head()

In [ ]:
print(df['bias_type'].value_counts())
print(df['label_name'].value_counts())

In [ ]:
# Load LLaMA-2 model
MODEL_NAME = 'meta-llama/Llama-2-7b-hf'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
 MODEL_NAME,
 torch_dtype=torch.float16,
 device_map='auto'
)
model.eval()
print('Model loaded:', MODEL_NAME)

In [ ]:
def compute_pll(sentence, tokenizer, model, device='cuda'):
 """
 Compute Pseudo-Log-Likelihood (PLL) for a sentence.
 PLL = sum of log P(w_i | w_j, j!=i) using masked token scoring.
 For causal LMs, approximated as: sum of log P(w_i | w_<i).
 """
 inputs = tokenizer(sentence, return_tensors='pt').to(device)
 input_ids = inputs['input_ids']
 with torch.no_grad():
 outputs = model(**inputs, labels=input_ids)
 # outputs.loss is the mean NLL per token
 n_tokens = input_ids.shape[1] - 1
 pll = -outputs.loss.item() * n_tokens
 return pll

In [ ]:
# Compute PLL for all English sentences
device = 'cuda' if torch.cuda.is_available() else 'cpu'
pll_scores = []
for _, row in tqdm(df.iterrows(), total=len(df)):
 pll = compute_pll(row['full_sentence'], tokenizer, model, device)
 pll_scores.append(pll)

df['PLLEnglish'] = pll_scores
df.head()

In [ ]:
# Compute BiasScore per context group
# BiasScore = PLL(stereotype) - PLL(anti-stereotype) for each context
bias_scores = []

for context, group in df.groupby('context'):
 stereo = group[group['label_name'] == 'stereotype']['PLLEnglish'].values
 anti = group[group['label_name'] == 'anti-stereotype']['PLLEnglish'].values
 if len(stereo) > 0 and len(anti) > 0:
 score = stereo[0] - anti[0]
 else:
 score = float('nan')
 for idx in group.index:
 if group.loc[idx, 'label_name'] != 'unrelated':
 bias_scores.append((idx, score))
 else:
 bias_scores.append((idx, float('nan')))

bias_df = pd.DataFrame(bias_scores, columns=['idx', 'BiasEnglish']).set_index('idx')
df['BiasEnglish'] = bias_df['BiasEnglish']

In [ ]:
# Compute Categorical Bias (CB) per bias type
# CB = mean PLL(stereotype) / (mean PLL(stereo) + mean PLL(anti) + mean PLL(unrelated))
print('--- Categorical Bias by Bias Type (LLaMA-2 English) ---')
for btype, group in df.groupby('bias_type'):
 pll_stereo = group[group['label_name']=='stereotype']['PLLEnglish'].mean()
 pll_anti = group[group['label_name']=='anti-stereotype']['PLLEnglish'].mean()
 pll_unrel = group[group['label_name']=='unrelated']['PLLEnglish'].mean()
 denom = abs(pll_stereo) + abs(pll_anti) + abs(pll_unrel)
 cb = (abs(pll_stereo) / denom) * 100
 print(f'{btype}: CB = {cb:.2f}')

In [ ]:
# Save results
df.to_csv('/content/drive/MyDrive/stereoset_LLama.csv', index=False)
print('Saved stereoset_LLama.csv')